In [1]:
import pyarrow as pa

with pa.memory_map("/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow", "r") as f:
    reader = pa.ipc.open_stream(f)
    table = reader.read_all()

print(table.schema)
print(table.num_rows)
print(table[0])

run_id: string
frame: int32
timestamp: float
image_front: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_front_left: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_front_right: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
image_rear: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
seg_front: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
lidar: list<item: list<item: float>>
  child 0, item: list<item: float>
      child 0, item: float
boxes: list<item: list<item: float>>
  child 0, item: list<item: float>
      child 0, item: float
box_labels: list<item: string>
  child 0, item: string
location_x: float
location_y: float
location_z: float
rotation_pitch: float
rotation_yaw: float
rotation_roll: float
velocity_x: float
velocity_y: float
velocity_z: float
speed_kmh: float
throttle: float


In [4]:
from PIL import Image
import io
import numpy as np

# bytes → PIL Image
img_bytes = table.to_pydict()["image_front"][0]["bytes"]
img = Image.open(io.BytesIO(img_bytes))
print(img.size)   # (W, H)
print(img.mode)   # RGB 확인

(800, 600)
RGBA


In [2]:
import numpy as np

row = table.to_pydict()
lidar_sample = row["lidar"][0]
lidar_np = np.array(lidar_sample)
print(lidar_np.shape)   # (N, ?) 확인
print(lidar_np[0])      # 첫 번째 포인트

(4856, 4)
[-7.43413773e+01  1.87355903e-12  1.31083908e+01  7.39372909e-01]


In [ ]:
d = table.to_pydict()

print(d["run_id"][:10])
print(d["frame"][:10])

['run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001']
[0, 5, 10, 15, 20, 25, 30, 35, 40, 45]


In [3]:
import numpy as np

def point_cloud_to_range_view(points, H=64, W=512):
    """
    points: (N, 4) - x, y, z, intensity
    returns: (4, H, W)
    """
    x, y, z, intensity = points[:, 0], points[:, 1], points[:, 2], points[:, 3]

    # 구면 좌표 변환
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw = np.arctan2(y, x)                    # [-π, π]
    pitch = np.arcsin(z / (r + 1e-8))         # [-π/2, π/2]

    # 정규화 → pixel 좌표
    yaw_img   = (yaw   / np.pi + 1) / 2       # [0, 1]
    pitch_img = (pitch / (np.pi/2) + 1) / 2   # [0, 1]

    col = (yaw_img   * (W - 1)).astype(int)
    row = ((1 - pitch_img) * (H - 1)).astype(int)

    col = np.clip(col, 0, W - 1)
    row = np.clip(row, 0, H - 1)

    range_view = np.zeros((4, H, W), dtype=np.float32)
    range_view[0, row, col] = x
    range_view[1, row, col] = y
    range_view[2, row, col] = z
    range_view[3, row, col] = intensity

    return range_view

# 테스트
rv = point_cloud_to_range_view(lidar_np)
print(rv.shape)   # (4, 64, 512)

(4, 64, 512)


In [5]:
img_rgb = img.convert("RGB")
print(img_rgb.mode)   # RGB
print(np.array(img_rgb).shape)   # (600, 800, 3)

RGB
(600, 800, 3)


In [6]:
from torchvision import transforms

to_tensor = transforms.ToTensor()
img_tensor = to_tensor(img_rgb)
print(img_tensor.shape)   # (3, 600, 800)

torch.Size([3, 600, 800])


In [8]:
import pyarrow as pa
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import io
import numpy as np
from torchvision import transforms

to_tensor = transforms.ToTensor()

def point_cloud_to_range_view(points, H=64, W=512):
    x, y, z, intensity = points[:, 0], points[:, 1], points[:, 2], points[:, 3]
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw   = np.arctan2(y, x)
    pitch = np.arcsin(z / (r + 1e-8))
    yaw_img   = (yaw   / np.pi + 1) / 2
    pitch_img = (pitch / (np.pi/2) + 1) / 2
    col = np.clip((yaw_img   * (W - 1)).astype(int), 0, W - 1)
    row = np.clip(((1 - pitch_img) * (H - 1)).astype(int), 0, H - 1)
    rv = np.zeros((4, H, W), dtype=np.float32)
    rv[0, row, col] = x
    rv[1, row, col] = y
    rv[2, row, col] = z
    rv[3, row, col] = intensity
    return rv

class MUVODataset(Dataset):
    def __init__(self, arrow_file, seq_len=4):
        self.seq_len = seq_len
        self.arrow_file = arrow_file

        with pa.memory_map(arrow_file, "r") as f:
            reader = pa.ipc.open_stream(f)
            table = reader.read_all()

        self.data = table.to_pydict()
        self.n = table.num_rows

        # 같은 run_id 안에서 연속 seq_len개 묶기
        self.index = []
        run_ids = self.data["run_id"]
        for i in range(self.n - seq_len + 1):
            if all(run_ids[i+j] == run_ids[i] for j in range(seq_len)):
                self.index.append(i)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        i = self.index[idx]
        images, lidars, actions, speeds = [], [], [], []

        for j in range(self.seq_len):
            k = i + j
            img = Image.open(io.BytesIO(self.data["image_front"][k]["bytes"])).convert("RGB")
            images.append(to_tensor(img))

            lidar_np = np.array(self.data["lidar"][k])
            lidars.append(torch.tensor(point_cloud_to_range_view(lidar_np)))

            actions.append([self.data["throttle"][k], self.data["steer"][k]])
            speeds.append(self.data["speed_kmh"][k])

        return {
            "image":  torch.stack(images),    # (S, 3, 600, 800)
            "lidar":  torch.stack(lidars),    # (S, 4, 64, 512)
            "action": torch.tensor(actions),  # (S, 2)
            "speed":  torch.tensor(speeds),   # (S,)
        }

# 테스트
arrow_file = "/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow"
ds = MUVODataset(arrow_file, seq_len=4)
print(f"dataset size: {len(ds)}")

sample = ds[0]
for k, v in sample.items():
    print(k, v.shape)

dataset size: 110
image torch.Size([4, 3, 600, 800])
lidar torch.Size([4, 4, 64, 512])
action torch.Size([4, 2])
speed torch.Size([4])


In [9]:
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=2, shuffle=True)
batch = next(iter(loader))

for k, v in batch.items():
    print(k, v.shape)
# image:  (2, 4, 3, 600, 800)
# lidar:  (2, 4, 4, 64, 512)
# action: (2, 4, 2)
# speed:  (2, 4)

image torch.Size([2, 4, 3, 600, 800])
lidar torch.Size([2, 4, 4, 64, 512])
action torch.Size([2, 4, 2])
speed torch.Size([2, 4])


In [10]:
import timm
import torch

# encoder 정의
encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4]
)
encoder.eval()

# (B, S, 3, 600, 800) → (B*S, 3, 600, 800)
B, S = 2, 4
image = batch["image"]                        # (2, 4, 3, 600, 800)
image_flat = image.view(B*S, 3, 600, 800)    # (8, 3, 600, 800)

xs = encoder(image_flat)
for i, x in enumerate(xs):
    print(f"scale {i}: {x.shape}")

scale 0: torch.Size([8, 128, 75, 100])
scale 1: torch.Size([8, 256, 38, 50])
scale 2: torch.Size([8, 512, 19, 25])


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from typing import List

class FPNDecoder(nn.Module):
    def __init__(self, feature_info, out_channels=128):
        super().__init__()
        # 가장 깊은 feature부터 시작
        self.conv1 = nn.Sequential(
            nn.Conv2d(feature_info[2], out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )
        self.skip_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(feature_info[i], out_channels, 3, 1, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(True),
            )
            for i in [1, 0]  # scale2, scale1 순서
        ])
        self.out_channels = out_channels

    def forward(self, xs: List[torch.Tensor]) -> torch.Tensor:
        x = self.conv1(xs[2])  # (B*S, out_channels, 19, 25)

        for i, conv in enumerate(self.skip_convs):
            size = xs[1 - i].shape[-2:]
            x = conv(xs[1 - i]) + F.interpolate(x, size=size, mode='bilinear', align_corners=False)

        return x  # (B*S, out_channels, 75, 100)

# 테스트
feature_info = [128, 256, 512]
fpn = FPNDecoder(feature_info, out_channels=128)
fpn.eval()

with torch.no_grad():
    Fc = fpn(xs)
    print(f"Fc: {Fc.shape}")  # (8, 128, 75, 100)

Fc: torch.Size([8, 128, 75, 100])


In [12]:
# LiDAR encoder (range-view 4채널 입력)
lidar_encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4],
    in_chans=4          # RGB 3채널 → LiDAR 4채널
)
lidar_encoder.eval()

lidar_fpn = FPNDecoder([128, 256, 512], out_channels=128)
lidar_fpn.eval()

# (B, S, 4, 64, 512) → (B*S, 4, 64, 512)
lidar_flat = batch["lidar"].view(B*S, 4, 64, 512)

with torch.no_grad():
    lidar_xs = lidar_encoder(lidar_flat)
    for i, x in enumerate(lidar_xs):
        print(f"lidar scale {i}: {x.shape}")
    
    FL = lidar_fpn(lidar_xs)
    print(f"FL: {FL.shape}")

lidar scale 0: torch.Size([8, 128, 8, 64])
lidar scale 1: torch.Size([8, 256, 4, 32])
lidar scale 2: torch.Size([8, 512, 2, 16])
FL: torch.Size([8, 128, 8, 64])


In [13]:
import math

class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=64, normalize=True):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.normalize = normalize
        self.scale = 2 * math.pi

    def forward(self, x):
        _, _, h, w = x.shape
        not_mask = torch.ones((1, h, w), device=x.device)
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            y_embed = y_embed / (y_embed[:, -1:, :] + 1e-6) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + 1e-6) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = 10000 ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:,:,:,0::2].sin(), pos_x[:,:,:,1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:,:,:,0::2].sin(), pos_y[:,:,:,1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)

d_model = 128
pos_enc = PositionEmbeddingSine(num_pos_feats=d_model // 2)

# type embedding: (1, d_model, 1, 1) 로 spatial broadcast 가능하게
type_embed_image = nn.Parameter(torch.zeros(1, d_model, 1, 1))
type_embed_lidar  = nn.Parameter(torch.zeros(1, d_model, 1, 1))

encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=8, dropout=0.1, batch_first=False)
transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)

# Fc를 AdaptiveAvgPool로 줄이기
pool = nn.AdaptiveAvgPool2d((12, 16))  # 7500 → 192 token

with torch.no_grad():
    Fc_small = pool(Fc)  # (8, 128, 12, 16)
    FL_small = pool(FL)  # (8, 128, 8, 14) → 크기 맞춰줌
    FL_small = nn.AdaptiveAvgPool2d((8, 16))(FL)  # (8, 128, 8, 16)

    print(f"image tokens: {Fc_small.shape}")  # (8, 128, 12, 16) → 192개
    print(f"lidar tokens: {FL_small.shape}")  # (8, 128,  8, 16) → 128개

    image_tokens = (Fc_small + pos_enc(Fc_small) + type_embed_image).flatten(2).permute(2, 0, 1)
    lidar_tokens = (FL_small + pos_enc(FL_small) + type_embed_lidar).flatten(2).permute(2, 0, 1)

    print(f"image tokens: {image_tokens.shape}")  # (192, 8, 128)
    print(f"lidar tokens: {lidar_tokens.shape}")  # (128, 8, 128)

    L_image = image_tokens.shape[0]
    tokens = torch.cat([image_tokens, lidar_tokens], dim=0)  # (320, 8, 128)
    print(f"tokens in: {tokens.shape}")

    tokens_out = transformer(tokens)
    print(f"tokens out: {tokens_out.shape}")

    image_tokens_out = tokens_out[:L_image]
    lidar_tokens_out = tokens_out[L_image:]
    print(f"image tokens out: {image_tokens_out.shape}")
    print(f"lidar tokens out: {lidar_tokens_out.shape}")

image tokens: torch.Size([8, 128, 12, 16])
lidar tokens: torch.Size([8, 128, 8, 16])
image tokens: torch.Size([192, 8, 128])
lidar tokens: torch.Size([128, 8, 128])
tokens in: torch.Size([320, 8, 128])
tokens out: torch.Size([320, 8, 128])
image tokens out: torch.Size([192, 8, 128])
lidar tokens out: torch.Size([128, 8, 128])


In [14]:
# tokens_out → 1D embedding
image_feat_conv = nn.Sequential(
    nn.Linear(d_model, d_model),
    nn.ReLU(),
    nn.AdaptiveAvgPool1d(1),  # (B*S, d_model, N) → (B*S, d_model, 1)
)

with torch.no_grad():
    # (N, B*S, C) → (B*S, C, N)
    image_out = image_tokens_out.permute(1, 2, 0)  # (8, 128, 192)
    lidar_out = lidar_tokens_out.permute(1, 2, 0)  # (8, 128, 128)

    image_embed = image_out.mean(dim=-1)  # (8, 128)
    lidar_embed = lidar_out.mean(dim=-1)  # (8, 128)

    # concat → linear로 fusion
    features_combine = nn.Linear(d_model * 2, d_model)
    embedding = features_combine(torch.cat([image_embed, lidar_embed], dim=-1))  # (8, 128)

    # (B*S, 128) → (B, S, 128)
    embedding = embedding.view(B, S, d_model)
    print(f"embedding: {embedding.shape}")  # (2, 4, 128)

embedding: torch.Size([2, 4, 128])


In [ ]:
# GRU: embedding 시퀀스 → hidden state
gru = nn.GRUCell(input_size=d_model, hidden_size=d_model)

with torch.no_grad():
    h_t = torch.zeros(B, d_model)  # 초기 hidden state

    hiddens = []
    for t in range(S):
        o_t = embedding[:, t, :]       # (B, 128)
        h_t = gru(o_t, h_t)            # (B, 128)
        hiddens.append(h_t)

    hiddens = torch.stack(hiddens, dim=1)  # (B, S, 128)
    print(f"hidden states: {hiddens.shape}")  # (2, 4, 128)